# L58 · 训练循环（training loop） — loss 计算、optimizer.step、收敛曲线（convergence curve），拟合 make_moons 数据集

**目标**：用自制 autograd 训练一个 MLP，拟合二维月牙形分类问题，画出损失曲线，彻底理解训练循环的每一步。

🔗 **Aurora 连接**：`src/aurora/` 中 Month 2 的 PyTorch 训练脚本与本节循环完全同构——`forward → loss → backward → update → zero_grad` 这五步在所有深度学习框架里一字不差；理解这个循环 = 理解所有深度学习训练。

训练神经网络本质上是一个迭代优化过程：每轮把数据喂进网络得到预测，用损失函数（loss function）衡量预测有多差，通过反向传播（backpropagation）算出每个参数对损失的偏导，然后沿梯度（gradient）反方向移动一小步。这个循环重复足够多轮后，参数会收敛到一个让损失最小的局部极值。本节用上一节自制的 `Value` 自动微分引擎和 `MLP`，端到端跑通整个流程。

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# 假设 a4_mlp.ipynb 中定义的 Value / MLP 已在同一 session 中执行
# 如果单独运行本 notebook，需先粘贴或 %run a4_mlp.ipynb
# 下面重新给出最小版本的 Value + MLP 定义（与 a4 完全一致）

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self): return f'Value({self.data:.4f})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def __rmul__(self, other): return self * other

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) - self
    def __truediv__(self, other): return self * Value(other)**-1 if not isinstance(other, Value) else self * other**-1

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp-1)) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()


class Neuron:
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1,1)) for _ in range(n_in)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum(wi*xi for wi,xi in zip(self.w,x)) + self.b
        return act.relu()
    def parameters(self): return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, n_in, layer_sizes):
        sizes = [n_in] + layer_sizes
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(layer_sizes))]
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x
    def parameters(self): return [p for l in self.layers for p in l.parameters()]

print('Value + MLP 定义完成 ✅')

## 1. 训练循环结构

标准训练循环五步，缺一不可：

```
for epoch in range(N):
    ypred = [model(x) for x in xs]   # forward
    loss  = compute_loss(ypred, ys)   # loss
    loss.backward()                   # backward
    for p in model.parameters():      # update
        p.data -= lr * p.grad
    for p in model.parameters():      # zero_grad
        p.grad = 0.0
```

每一步的作用：**forward** 产生预测；**loss** 把「有多差」压缩成一个标量；**backward** 把这个标量的梯度反向传播到每个参数；**update** 沿梯度反方向微移参数；**zero_grad** 清除本轮梯度，防止下轮累积。

In [ ]:
# 演示：一个最简单的一步训练循环
random.seed(0)
tiny_model = MLP(2, [1])        # 输入2维，输出1个标量
x_demo = [Value(0.5), Value(-0.3)]
y_demo = 1.0                     # 目标类别 +1

score = tiny_model(x_demo)
hinge = (Value(1.0) - Value(y_demo) * score).relu()   # max(0, 1 - y*score)
print(f'score={score.data:.4f}  hinge={hinge.data:.4f}')
hinge.backward()
print('参数梯度（前5个）:', [f'{p.grad:.4f}' for p in tiny_model.parameters()[:5]])
# 做一步更新
lr = 0.05
for p in tiny_model.parameters(): p.data -= lr * p.grad
for p in tiny_model.parameters(): p.grad = 0.0
print('一步更新完成 ✅')

## 2. Hinge Loss（SVM 风格损失）

二分类时，标签 `y ∈ {+1, -1}`，网络输出一个实数分数 `score`。

Hinge loss 定义：
```
L_i = max(0,  1 - y_i * score_i)
```

当预测方向正确且置信度足够高时（`y*score >= 1`），损失为 0；否则损失线性增长。

批量均值损失：
```
L = mean(L_i  for  i in range(N))
```

与交叉熵相比，hinge loss 更稀疏（大部分正确样本梯度为 0），适合 SVM 和对比学习场景。本节选它是因为它只用加法和 `relu`，完全在自制 `Value` 的支持范围内。

In [ ]:
# 可视化 hinge loss 随 y*score 的变化
margin = np.linspace(-1.5, 2.5, 200)
hinge_vals = np.maximum(0, 1 - margin)
plt.figure(figsize=(5,3))
plt.plot(margin, hinge_vals, lw=2, color='steelblue')
plt.axvline(1, color='gray', ls='--', lw=1, label='margin=1')
plt.xlabel('y × score')
plt.ylabel('hinge loss')
plt.title('Hinge Loss = max(0, 1 - y·score)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. zero_grad：为什么每轮必须清零？

`Value.backward()` 的实现里，梯度是**累加**到 `.grad` 上的（`self.grad += ...`）。这是为了支持一个节点被多条路径引用时梯度正确叠加。

但跨 epoch 时，`.grad` 不会自动归零。如果不手动清零：
```
epoch1: p.grad = 0.3
epoch2: p.grad = 0.3 + 0.3 = 0.6   # 错！
epoch3: p.grad = 0.9               # 越来越大，参数更新爆炸
```

所以每轮 `backward` 之前（或 `update` 之后）必须执行：
```python
for p in model.parameters():
    p.grad = 0.0
```
这在 PyTorch 里对应 `optimizer.zero_grad()`，原理完全相同。

In [ ]:
# 演示梯度累积的危害
random.seed(42)
m = MLP(2, [1])
x = [Value(1.0), Value(0.5)]
y = 1.0

print('不清零时 .grad 的变化：')
for i in range(3):
    s = m(x)
    loss = (Value(1.0) - Value(y) * s).relu()
    loss.backward()   # 不清零
    grad_0 = m.parameters()[0].grad
    print(f'  epoch {i+1}: params[0].grad = {grad_0:.4f}')

print('\n清零后 .grad 的变化：')
for p in m.parameters(): p.grad = 0.0   # 先清零一次
for i in range(3):
    s = m(x)
    loss = (Value(1.0) - Value(y) * s).relu()
    loss.backward()
    grad_0 = m.parameters()[0].grad
    print(f'  epoch {i+1}: params[0].grad = {grad_0:.4f}')
    for p in m.parameters(): p.grad = 0.0   # 每轮清零

## 4. ✏️ 实现 `train(model, xs, ys, lr=0.05, epochs=100)`

**推理路线**：
1. 外层 `for epoch in range(epochs)` 控制轮数
2. 对每个样本做 forward：`ypred = [model(x) for x in xs]`
3. 计算每个样本的 hinge loss：`(Value(1.0) - Value(y)*pred).relu()`
4. 取均值：`loss = sum(losses) * (1.0/len(losses))`
5. 反向传播：`loss.backward()`
6. 参数更新：`p.data -= lr * p.grad`
7. 清零：`p.grad = 0.0 for p in model.parameters()`
8. 把 `loss.data` 记录到列表，最终返回

**参考输入输出**：
- 输入：月牙形双类 200 个点（`make_moons`），标签 `±1`，`lr=0.05`，`epochs=100`
- 输出：长度 100 的 float 列表，`losses[0] ≈ 0.8`，`losses[-1] < 0.2`，最终分类准确率 > 85%

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def train(model, xs, ys, lr=0.05, epochs=100):
    """训练 model，返回每轮的 loss 值列表。"""
    loss_history = []
    for epoch in range(epochs):
        # ✏️ TODO: forward — 对所有样本计算预测
        ypred = None  # 替换此行

        # ✏️ TODO: 计算每个样本的 hinge loss = max(0, 1 - y*pred)
        losses = None  # 替换此行

        # ✏️ TODO: 取均值得到标量 loss
        loss = None   # 替换此行

        # ✏️ TODO: 反向传播

        # ✏️ TODO: 参数更新（p.data -= lr * p.grad）

        # ✏️ TODO: 清零梯度（p.grad = 0.0）

        loss_history.append(loss.data)
        if (epoch + 1) % 20 == 0:
            print(f'epoch {epoch+1:3d}  loss={loss.data:.4f}')
    return loss_history

In [ ]:
# ── 生成月牙形数据集 ──
try:
    from sklearn.datasets import make_moons
    _HAS_SKLEARN = True
except ImportError:
    _HAS_SKLEARN = False
    print("sklearn 未安装，请先：pip install scikit-learn")
if not _HAS_SKLEARN:
    raise ImportError("请先安装 scikit-learn：pip install scikit-learn") from None
X_raw, Y_raw = make_moons(n_samples=200, noise=0.15, random_state=7)
# 标签转为 ±1
Ys = [2*y-1 for y in Y_raw.tolist()]
# 转为 Value 列表
Xs = [[Value(x) for x in row] for row in X_raw.tolist()]

# 初始化 MLP：输入2维，隐层16+16，输出1个标量
random.seed(0)
model = MLP(2, [16, 16, 1])
print(f'参数总数：{len(model.parameters())}')

# ── 训练 ──
history = train(model, Xs, Ys, lr=0.05, epochs=100)

# ── 验收 ──
with_pred = [model([Value(x) for x in row]).data for row in X_raw.tolist()]
acc = sum(1 for p,y in zip(with_pred, Ys) if (p>=0) == (y==1)) / len(Ys)
print(f'\n最终准确率：{acc*100:.1f}%')
assert history[-1] < history[0], '损失应下降'
assert acc > 0.85, f'准确率 {acc:.2f} 未超过 85%'
print('✅ 训练验收通过')

# ── 画损失曲线 ──
plt.figure(figsize=(5,3))
plt.plot(history, lw=2, color='steelblue')
plt.xlabel('epoch')
plt.ylabel('hinge loss')
plt.title('训练损失曲线（lr=0.05）')
plt.tight_layout()
plt.show()

# ── 目视验证：决策边界 ──
h = 0.05
xx, yy = np.meshgrid(np.arange(X_raw[:,0].min()-0.3, X_raw[:,0].max()+0.3, h),
                      np.arange(X_raw[:,1].min()-0.3, X_raw[:,1].max()+0.3, h))
Z = np.array([model([Value(float(a)), Value(float(b))]).data
              for a,b in zip(xx.ravel(), yy.ravel())]).reshape(xx.shape)
plt.figure(figsize=(5,4))
plt.contourf(xx, yy, Z, levels=0, alpha=0.3, colors=['#f77', '#77f'])
plt.scatter(X_raw[:,0], X_raw[:,1], c=['red' if y==1 else 'blue' for y in Ys], s=15)
plt.title('决策边界（目视验证）')
plt.tight_layout()
plt.show()

## 5. 参数实验：学习率对收敛的影响

分别用 `lr=0.001`、`lr=0.05`、`lr=0.5` 训练 150 轮，画三条 loss 曲线：
- `lr=0.001`：步长太小，损失下降极慢，150 轮后仍未收敛
- `lr=0.05`：稳定收敛，loss 平滑下降
- `lr=0.5`：步长过大，loss 震荡剧烈，可能发散

In [ ]:
EPOCHS = 150
results = {}
for lr_val in [0.001, 0.05, 0.5]:
    random.seed(0)
    m = MLP(2, [16, 16, 1])
    print(f'\n── lr={lr_val} ──')
    h = train(m, Xs, Ys, lr=lr_val, epochs=EPOCHS)
    results[lr_val] = h

plt.figure(figsize=(6,4))
colors = {0.001:'orange', 0.05:'steelblue', 0.5:'crimson'}
for lr_val, h in results.items():
    plt.plot(h, label=f'lr={lr_val}', color=colors[lr_val], lw=1.8)
plt.xlabel('epoch')
plt.ylabel('hinge loss')
plt.title('学习率对收敛速度/稳定性的影响')
plt.legend()
plt.ylim(0, max(results[0.5][:10])*1.1 if max(results[0.5][:5])>1 else 1.5)
plt.tight_layout()
plt.show()

## 本课收束

`train()` 函数实现了完整的梯度下降（gradient descent，GD）训练循环：`forward → hinge loss → backward → update → zero_grad`，返回每轮的损失数值列表。月牙形数据集上 100 轮后准确率超过 85%，损失曲线验证了收敛正常。这个循环与 `src/aurora/` 中 Month 2 PyTorch 版本完全同构——参数更新逻辑、zero_grad 时机、loss 聚合方式一字不差。下一节（L59）将进入 PyTorch 世界：从 `torch.Tensor` 开始，掌握张量操作、dtype 转换与设备管理，为后续 nn.Module 构建做好基础。ch）随机梯度下降。